## Notebook to compare single neuron morphologies with MAPseq dataset
Start by establishing dataframe

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import lc_reconstruction_analysis.utils as utils
import lc_reconstruction_analysis.dendritic as dendritic
import lc_reconstruction_analysis.clustering as clustering
import lc_reconstruction_analysis.axon as axon
import lc_reconstruction_analysis.correlation as correlation
%matplotlib inline

In [2]:
# Define path to data
DATA_DIR = Path("/data/")
# Load data frame of cells, and networkx graphs
dataDF, graphs = utils.load_cells(DATA_DIR)

Missing structure dendrite for /data/685221_12_5_24/Complete_annotated/N064-68221-JN.json
Missing structure dendrite for /data/685221_12_5_24/Complete_annotated/N025-685221-PG.json
Missing structure axon for /data/685221_12_5_24/Complete_annotated/N064-685221-JN.json
Missing structure dendrite for /data/685221_12_5_24/Complete_annotated/N051-685221 -YP.json
Missing structure dendrite for /data/648434_12_5_24/Complete_annotated/N024-648434-SS.json
Error finding structures for: N025-685221-PG, dropping from dataframe
Error finding structures for: N064-68221-JN, dropping from dataframe
Error finding structures for: N064-685221-JN, dropping from dataframe
Error finding structures for: N051-685221 -YP, dropping from dataframe
Error finding structures for: N024-648434-SS, dropping from dataframe


In [3]:
# Exclude any neurons that need to be excluded
excludeList = [ # Look like MY cells mistakenly included in LC batch
    "N039-685222-AK",
    "N040-685222-VM",
    "N066-685221-JN",
    "N068-685221-HD",
    "N067-685221-HS"
]

dataDF = dataDF.drop(dataDF[dataDF["Graph"].isin(excludeList)].index)

In [4]:
# Load data
df = clustering.build_length_df(dataDF, graphs, DATA_DIR, normalize_df=False)
df = df.drop(list(set(df.index) - set(dataDF["Graph"]))) # Drop graphs not included in dataDF
# Normalize length within each neuron
dfNorm = df.copy()
dfNorm = dfNorm.divide(dfNorm.sum(axis=1),axis=0)
# Merge dataframes
sorted_columns = ["OLF","Isocortex","HPF","CTXsp","CNU","TH","HY","MB","CB","P","MY","Other"]
plotDF = dataDF.copy().merge(dfNorm[sorted_columns], left_on="Graph", right_index = True)

In [5]:
# Add total length
plotDF = plotDF.merge(pd.Series(df.sum(axis=1) / 1e4 , name = "Total Length (cm)"), left_on="Graph",right_index = True) # convert from microns to centimeters
plotDF["Norm Length"] = plotDF["Total Length (cm)"] / plotDF["Total Length (cm)"].max()

# Compute total axonal branches
axon_branch_dict = {}
for name in plotDF["Graph"]:
    graph = graphs[name]
    axon_branch_nodes = [node for node in graph.nodes() if graph.out_degree(node) > 1 and graph.nodes[node]["structure_id"] == 2]
    axon_branch_dict[name] = len(axon_branch_nodes)

# Add axonal branches to dataframe
plotDF = plotDF.merge(pd.Series(axon_branch_dict, name = "Axon Branches"), left_on = "Graph", right_index = True)
plotDF["Norm Branches"] = plotDF["Axon Branches"] / plotDF["Axon Branches"].max()

In [6]:
plotDF.head(3)

,Graph,ID,Sample,Annotator,Genotype,somaAP,somaDV,somaML,somaOnRight,OLF,...,HY,MB,CB,P,MY,Other,Total Length (cm),Norm Length,Axon Branches,Norm Branches
5,N001-685221-PG,N001,685221,PG,Dbh-Cre-KI/wt,10201.6760,4571.1625,4612.8491,False,0.000000,...,0.020860,0.255037,0.000000,0.071944,0.338564,0.173288,14.318932,0.197319,103,0.066796
62,N001-685222-SA,N001,685222,SA,Dbh-Cre-KI/wt,10345.2172,4145.2136,4709.6842,True,0.047268,...,0.028757,0.040161,0.006982,0.004348,0.000000,0.009400,72.567420,1.000000,982,0.636835
63,N002-685222-HD,N002,685222,HD,Dbh-Cre-KI/wt,10559.8164,4318.4146,4715.9901,True,0.398272,...,0.018671,0.015920,0.004814,0.007940,0.000465,0.077648,49.870732,0.687233,809,0.524643


### Get subset of graphs within CCF bounds

In [ ]:
# Get CCF lookups
from collections import defaultdict
ccf_structures = pd.read_csv('/data/allen_mouse_ccf/annotation/adult_mouse_ccf_structures.csv')
id_to_acronym = defaultdict(lambda: "NaN")
id_to_acronym = id_to_acronym | ccf_structures.set_index('id')['acronym'].to_dict()
acronym_to_id = {acronym: ccf_id for ccf_id, acronym in id_to_acronym.items()}
id_to_parent = ccf_structures.set_index('id')['parent_structure_id'].to_dict()
id_to_parent[None] = None # to account for out of bands points

In [ ]:
# Create pathing dictionaries to group children CCF levels
rois = ["CB","MY","P","MB","TH","HY","CNU","CTXsp","HPF","OLF","Isocortex","fiber tracts","VS","grv","retina"] # remove "fiber tracts","VS","grv","retina" later
roiIDs = [acronym_to_id[roi] for roi in rois]
# Get the pathing for each leaf node, find which roi each belongs to
id_to_path = ccf_structures.set_index("id")["structure_id_path"].to_dict()
# Create a new dictionary corresponding each CCF copartment to matching ROI
# id_to_roi = {}
id_to_roi = defaultdict(lambda: np.nan)
# For each CCF compartment, break down path and find matching ROI
for key, val in id_to_path.items():
    # Break down path 
    pathList = [int(struct) for struct in val.split("/") if struct]
    # Find matching ROI (should be empty, or a single element)
    roiList = [id for id in roiIDs if id in pathList]
    if not roiList:
        id_to_roi[key] = np.nan
    else:
        id_to_roi[key] = roiList[0]

# To check which values are uanccounted for
# [id_to_acronym[struct] for struct in [key for key, val in id_to_roi.items() if np.isnan(val)]]

In [94]:
graphs["N018-685221-DS"].nodes[383]

{'pos': (10479.6081, 4475.7638, 4580.1781),
 'radius': 1.0,
 'structure_id': 3,
 'allen_id': 771}

In [ ]:
for i, graph in graphs.items():
    # Assemble all nodes and edges
    nodes = graph.nodes(data=True)
    edges = graph.edges()
        
    regions = {node['allen_id'] for _, node in nodes}
    region_lengths = {}